# Notebook 3: The Agentic Loop — The Core of Your Interview

**This is the most important notebook.** The agentic loop is what turns a single tool call into a multi-step autonomous agent. This is what you'll build in the interview.

By the end, you'll be able to:
- Write the `while` loop that lets Claude use tools until it's done
- Handle sequential tool calls (tool A → tool B → answer)
- Handle parallel tool calls within the loop
- Add safety limits (max iterations)
- Debug the loop when things go wrong

---

In [ ]:
!pip install anthropic -q
import anthropic
import json

client = anthropic.Anthropic()

## 3.1 Why a Loop?

In Notebook 2, we handled ONE round of tool use. But real agents need **multiple rounds**:

```
User: "What's the weather where I am?"

Round 1: Claude calls get_location() → "San Francisco, CA"
Round 2: Claude calls get_weather("San Francisco, CA") → "62F, foggy"
Round 3: Claude responds: "It's 62F and foggy in San Francisco."
```

Claude needed TWO tool calls sequentially, where the output of the first feeds into the second. Without a loop, you'd need to hardcode each step.

### The Core Insight
**Keep calling the API in a loop until `stop_reason` is NOT `"tool_use"`.**

## 3.2 The Minimal Agentic Loop

Here's the simplest possible version. **Memorize this pattern.**

```python
messages = [{"role": "user", "content": user_message}]

while True:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=4096,
        tools=tools,
        messages=messages
    )
    
    # If Claude is done, break out of the loop
    if response.stop_reason == "end_turn":
        break
    
    # Claude wants to use tools — process them
    messages.append({"role": "assistant", "content": response.content})
    
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = process_tool_call(block.name, block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result)
            })
    
    messages.append({"role": "user", "content": tool_results})

# After the loop, response contains the final answer
final_text = response.content[0].text
```

That's it. 20 lines. This is your interview answer.

In [ ]:
# ========== Setup: Tools and handlers ==========

def get_location():
    """Simulates getting user's location from IP."""
    return json.dumps({"city": "San Francisco", "state": "CA", "country": "US"})

def get_weather(location, unit="fahrenheit"):
    """Fake weather API."""
    return json.dumps({"location": location, "temp": 62, "unit": unit, "condition": "foggy"})

def calculator(operation, a, b):
    """Basic calculator."""
    ops = {"add": a+b, "subtract": a-b, "multiply": a*b, "divide": a/b if b else "div/0"}
    return str(ops.get(operation, "unknown op"))

def get_time(timezone):
    """Fake time lookup."""
    return json.dumps({"timezone": timezone, "time": "2:30 PM PST"})

# Tool definitions
tools = [
    {
        "name": "get_location",
        "description": "Get the current user's location based on IP address. Takes no parameters.",
        "input_schema": {"type": "object", "properties": {}}
    },
    {
        "name": "get_weather",
        "description": "Get current weather for a location. Returns temp, conditions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and state, e.g. San Francisco, CA"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["location"]
        }
    },
    {
        "name": "calculator",
        "description": "Perform arithmetic: add, subtract, multiply, divide.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["operation", "a", "b"]
        }
    },
    {
        "name": "get_time",
        "description": "Get current time in a timezone. Use IANA timezone names.",
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {"type": "string", "description": "IANA timezone, e.g. America/Los_Angeles"}
            },
            "required": ["timezone"]
        }
    }
]

# Dispatcher
handlers = {
    "get_location": get_location,
    "get_weather": get_weather,
    "calculator": calculator,
    "get_time": get_time,
}

def process_tool_call(name, input_data):
    handler = handlers.get(name)
    if not handler:
        return f"Unknown tool: {name}"
    try:
        return handler(**input_data)
    except Exception as e:
        return f"Error: {e}"

print("Setup complete.")

In [ ]:
# ========== THE AGENTIC LOOP ==========

def agent_loop(user_message, tools, process_tool_call, max_iterations=10):
    """
    The core agentic loop. This is what you build in the interview.
    
    Args:
        user_message: The user's question
        tools: List of tool definitions
        process_tool_call: Function that executes a tool call
        max_iterations: Safety limit to prevent infinite loops
    
    Returns:
        Final text response from Claude
    """
    messages = [{"role": "user", "content": user_message}]
    
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # Call the API
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        print(f"Stop reason: {response.stop_reason}")
        
        # If Claude is done, return the final text
        if response.stop_reason == "end_turn":
            final_text = next(
                (b.text for b in response.content if b.type == "text"), 
                "No text response"
            )
            print(f"DONE: {final_text[:100]}...")
            return final_text
        
        # Claude wants to use tools
        if response.stop_reason == "tool_use":
            # Add Claude's response to message history
            messages.append({"role": "assistant", "content": response.content})
            
            # Process all tool calls
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool: {block.name}({json.dumps(block.input)})")
                    result = process_tool_call(block.name, block.input)
                    print(f"  Result: {result[:100]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            # Add tool results to message history
            messages.append({"role": "user", "content": tool_results})
    
    return "Max iterations reached"

print("Agent loop defined.")

## 3.3 Test: Sequential Tool Calls

This query requires Claude to first get location, then use that to get weather. Two sequential tool calls.

In [ ]:
# Sequential: get_location → get_weather
result = agent_loop(
    "What's the weather like where I am?",
    tools,
    process_tool_call
)
print(f"\n{'='*60}")
print(f"FINAL ANSWER: {result}")

### What just happened:

1. **Iteration 1**: Claude calls `get_location()` → returns San Francisco, CA
2. **Iteration 2**: Claude calls `get_weather("San Francisco, CA")` → returns 62F, foggy
3. **Iteration 3**: Claude composes a final answer using both results

The loop handled this automatically. Claude decided what tools to call and in what order.

In [ ]:
# Simple single-tool call (should be just 2 iterations)
result = agent_loop(
    "What's 42 * 37?",
    tools,
    process_tool_call
)
print(f"\nFINAL: {result}")

In [ ]:
# No tools needed (should be just 1 iteration)
result = agent_loop(
    "What is the capital of France?",
    tools,
    process_tool_call
)
print(f"\nFINAL: {result}")

## 3.4 The Clean Version (Interview-Ready)

Here's the same loop without all the debug prints. This is what you'd write in the interview.

In [ ]:
def run_agent(user_message, tools, process_tool_call, max_turns=10):
    """Clean agentic loop — the interview version."""
    messages = [{"role": "user", "content": user_message}]
    
    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        # Done — return final answer
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")
        
        # Tool use — execute and continue
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = process_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"

# Test
print(run_agent("What's the weather where I am?", tools, process_tool_call))

## 3.5 Anatomy of the Loop — What You Must Explain in the Discussion

After coding, the interviewer will ask about your design decisions. Here's what to say:

### Q: "Why a while/for loop?"
**A**: Claude may need multiple sequential tool calls to solve a problem. Each iteration, Claude decides whether to call another tool or give a final answer. The loop lets Claude drive the reasoning.

### Q: "What's the termination condition?"
**A**: `stop_reason == "end_turn"` means Claude has composed its final answer. The `max_turns` parameter is a safety valve to prevent infinite loops (e.g., if the model gets stuck in a retry cycle).

### Q: "Why do you append response.content directly?"
**A**: The API expects the exact content blocks it generated in previous turns. Modifying them could break the conversation. `response.content` is already in the right format.

### Q: "How do you handle parallel vs sequential tool calls?"
**A**: The loop handles both. If Claude returns multiple `tool_use` blocks in one response, I process all of them and return all results in a single `user` message. If Claude returns one, I process just that one. The code is the same either way.

### Q: "What about error handling?"
**A**: If a tool fails, I return the error as a `tool_result` with `is_error: true`. Claude will see the error and can decide to retry, try a different approach, or tell the user about the error.

## 3.6 Adding System Prompts

System prompts shape Claude's behavior. In your interview, you might want to add one to make the agent more focused.

In [ ]:
def run_agent_with_system(user_message, tools, process_tool_call, system_prompt=None, max_turns=10):
    """Agentic loop with optional system prompt."""
    messages = [{"role": "user", "content": user_message}]
    
    # Build API kwargs
    api_kwargs = {
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 4096,
        "tools": tools,
        "messages": messages,
    }
    if system_prompt:
        api_kwargs["system"] = system_prompt
    
    for _ in range(max_turns):
        response = client.messages.create(**api_kwargs)
        
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = process_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"

# Test with a system prompt
result = run_agent_with_system(
    "What's the weather where I am?",
    tools,
    process_tool_call,
    system_prompt="You are a concise weather assistant. Give brief, factual responses."
)
print(result)

## 3.7 Exercise: Build an Agent from Scratch

**Scenario**: Build a research assistant agent with these tools:

1. `search_articles` — takes `query` (string), returns a list of article titles and IDs
2. `get_article` — takes `article_id` (string), returns the article's full text
3. `summarize_text` — takes `text` (string), `max_words` (integer), returns a summary

The agent should be able to: search for articles, read the top result, and summarize it.

Write:
1. The tool definitions
2. The handler functions (fake data is fine)
3. The agentic loop
4. Test with: "Find an article about climate change and give me a brief summary"

In [ ]:
# YOUR ANSWER: Build the complete agent here

# 1. Tool handler functions

# 2. Tool definitions

# 3. Dispatcher

# 4. Agent loop

# 5. Test


In [ ]:
# SOLUTION

# 1. Handler functions
def search_articles(query):
    """Fake article search."""
    return json.dumps({
        "results": [
            {"id": "ART-001", "title": "Climate Change: A 2025 Overview", "relevance": 0.95},
            {"id": "ART-002", "title": "Rising Sea Levels and Coastal Cities", "relevance": 0.87},
            {"id": "ART-003", "title": "Renewable Energy Trends", "relevance": 0.72},
        ]
    })

def get_article(article_id):
    """Fake article retrieval."""
    articles = {
        "ART-001": {
            "title": "Climate Change: A 2025 Overview",
            "text": (
                "Global temperatures have risen by 1.2 degrees Celsius above pre-industrial levels. "
                "The Arctic is warming twice as fast as the global average. Sea levels have risen "
                "by approximately 20cm since 1900. Extreme weather events including hurricanes, "
                "droughts, and floods have increased in frequency. International agreements aim "
                "to limit warming to 1.5 degrees, but current policies project 2.7 degrees by 2100. "
                "Carbon dioxide levels reached 420 ppm in 2024, the highest in 800,000 years."
            )
        }
    }
    article = articles.get(article_id, {"error": "Article not found"})
    return json.dumps(article)

def summarize_text(text, max_words=50):
    """Fake summarizer (just truncates for demo)."""
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + "..."

# 2. Tool definitions
research_tools = [
    {
        "name": "search_articles",
        "description": (
            "Search for academic/news articles matching a query. Returns a list of "
            "articles with their IDs, titles, and relevance scores. Use this first "
            "to find relevant articles before reading them."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_article",
        "description": (
            "Retrieve the full text of an article by its ID. Use after search_articles "
            "to read the content of a specific article."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "article_id": {"type": "string", "description": "The article ID, e.g. ART-001"}
            },
            "required": ["article_id"]
        }
    },
    {
        "name": "summarize_text",
        "description": (
            "Generate a concise summary of a given text. Use after retrieving an article "
            "to create a digestible summary for the user."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to summarize"},
                "max_words": {"type": "integer", "description": "Maximum words in summary. Default 50."}
            },
            "required": ["text"]
        }
    }
]

# 3. Dispatcher
research_handlers = {
    "search_articles": search_articles,
    "get_article": get_article,
    "summarize_text": summarize_text,
}

def process_research_tool(name, input_data):
    handler = research_handlers.get(name)
    if not handler:
        return f"Unknown tool: {name}"
    try:
        return handler(**input_data)
    except Exception as e:
        return f"Error: {e}"

# 4. Run the agent
result = agent_loop(
    "Find an article about climate change and give me a brief summary.",
    research_tools,
    process_research_tool
)

print(f"\n{'='*60}")
print(f"FINAL: {result}")

## 3.8 Debugging the Loop

Things that can go wrong in the interview and how to fix them:

### Problem 1: Infinite loop
**Symptom**: Claude keeps calling tools forever
**Fix**: Always have `max_iterations` / `max_turns`

### Problem 2: API error about message format
**Symptom**: 400 error about "tool_use ids were found without tool_result blocks"
**Fix**: Make sure every `tool_use` gets a matching `tool_result` with the right `tool_use_id`

### Problem 3: Claude doesn't use the right tool
**Symptom**: Claude calls the wrong tool or makes up parameters
**Fix**: Improve the tool `description`. Be more specific about when to use each tool.

### Problem 4: "content" type error
**Symptom**: Content must be string or list error
**Fix**: Make sure `process_tool_call` returns a **string**, not a dict or None

In [ ]:
# Debugging helper: print the full message history

def print_messages(messages):
    """Pretty-print the message history for debugging."""
    for i, msg in enumerate(messages):
        role = msg["role"]
        content = msg["content"]
        
        print(f"\n[Message {i}] role={role}")
        
        if isinstance(content, str):
            print(f"  text: {content[:100]}")
        elif isinstance(content, list):
            for j, block in enumerate(content):
                if hasattr(block, 'type'):  # API response object
                    if block.type == "text":
                        print(f"  [{j}] text: {block.text[:80]}")
                    elif block.type == "tool_use":
                        print(f"  [{j}] tool_use: {block.name} id={block.id}")
                elif isinstance(block, dict):  # Our constructed message
                    btype = block.get("type", "?")
                    if btype == "tool_result":
                        print(f"  [{j}] tool_result: id={block['tool_use_id']} content={str(block.get('content',''))[:50]}")
                    elif btype == "text":
                        print(f"  [{j}] text: {block.get('text', '')[:80]}")

print("Debug helper defined. Use print_messages(messages) when debugging.")

## 3.9 Complete Template — Copy This Into the Interview

Here's the complete, minimal template you can start from in the interview:

In [ ]:
# ============================================
# INTERVIEW TEMPLATE: Complete Agent
# ============================================

import anthropic
import json

client = anthropic.Anthropic()

# ---- 1. DEFINE TOOL HANDLERS ----
def my_tool_1(param1, param2="default"):
    """Your tool implementation here."""
    return json.dumps({"result": "..."})

# ---- 2. DEFINE TOOL SCHEMAS ----
tools = [
    {
        "name": "my_tool_1",
        "description": "Detailed description of what this tool does...",
        "input_schema": {
            "type": "object",
            "properties": {
                "param1": {"type": "string", "description": "..."},
                "param2": {"type": "string", "description": "..."},
            },
            "required": ["param1"]
        }
    }
]

# ---- 3. DISPATCHER ----
TOOL_HANDLERS = {
    "my_tool_1": my_tool_1,
}

def process_tool_call(name, tool_input):
    handler = TOOL_HANDLERS.get(name)
    if not handler:
        return f"Unknown tool: {name}"
    try:
        return handler(**tool_input)
    except Exception as e:
        return f"Error: {e}"

# ---- 4. AGENTIC LOOP ----
def run_agent(user_message, max_turns=10):
    messages = [{"role": "user", "content": user_message}]
    
    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = process_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"

# ---- 5. RUN ----
# print(run_agent("Your test query here"))

---

## Summary — The Agentic Loop in 30 Seconds

1. Start with `messages = [{"role": "user", "content": user_msg}]`
2. Loop: call API → check `stop_reason`
3. If `"end_turn"` → done, return text
4. If `"tool_use"` → append assistant content, execute tools, append tool_results, continue loop
5. Safety: `max_turns` to prevent infinite loops

**This loop is the heart of your interview. Practice writing it from memory.**

**Next: Notebook 4 — Advanced Patterns (state, multi-turn conversations, complex scenarios)**